# NB02: BTFR Covariance Test -- SPARC Resolved Sample

Tests whether the g(Rt) ~ a0/2 alignment is an artifact of the baryonic Tully-Fisher
relation (BTFR). Uses a **fixed-slope** trend removal with Paper 2's calibration --
no parameters are fit to this sample.

**Locked pre-registration values:**
- Slope alpha = 0.238 (from Paper 2 external calibration)
- Intercept beta = -12.55 (anchored so Paper 2 SPARC median falls at a0/2)

**Expected outcome:**
- Raw scatter = 0.409 dex
- BTFR-residual scatter = 0.277 dex (32% reduction)
- Wilcoxon test: not significant (residuals centered near zero by construction)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.fit import run_phase1_baseline
from src.btfr import (
    run_btfr_covariance_test,
    compute_mbar_for_sample,
    BTFR_ALPHA,
    BTFR_BETA,
)
from src.physics import A0_HALF, ACCEL_TO_MKS

a0_half_mks = A0_HALF * ACCEL_TO_MKS

results_dir = project_root / "results"
results_dir.mkdir(exist_ok=True)

## 1. Load Baseline Data

In [ ]:
baseline_df = run_phase1_baseline()
galaxy_ids = baseline_df["galaxy_id"].tolist()
print(f"Resolved galaxies: {len(galaxy_ids)}")

## 2. Compute Baryonic Mass

In [ ]:
mbar_dict = compute_mbar_for_sample(galaxy_ids)

# Merge into baseline dataframe
baseline_df["m_bar"] = baseline_df["galaxy_id"].map(mbar_dict)
merged = baseline_df.dropna(subset=["m_bar"]).copy()

median_mbar = np.median(merged["m_bar"])
print(f"Galaxies with M_bar: {len(merged)} / {len(baseline_df)}")
print(f"Median M_bar:        {median_mbar:.3e} Msun")
print(f"Expected:            5.80e+09 Msun")

## 3. BTFR Trend (Fixed Slope)

The BTFR trend line is defined by:

$$\log_{10}(g_{\text{trend}}) = \alpha \cdot \log_{10}(M_{\text{bar}}) + \beta$$

where alpha = 0.238 and beta = -12.55 are **locked from pre-registration**.

The intercept beta was derived as: `beta = log10(a0/2) - alpha * log10(5.80e9)` so that
the Paper 2 SPARC median sits on the trend line at a0/2.

## 4. Run Covariance Test

In [ ]:
g_array = merged["g_Rt_mks"].values
mbar_array = merged["m_bar"].values

result = run_btfr_covariance_test(g_array, mbar_array)

print("BTFR Covariance Test Results")
print("=" * 50)
print(f"  N galaxies:          {result['n_galaxies']}")
print(f"  Raw scatter:         {result['scatter_raw']:.3f} dex")
print(f"  Residual scatter:    {result['scatter_residual']:.3f} dex")
print(f"  Scatter reduction:   {(1 - result['scatter_residual'] / result['scatter_raw']) * 100:.0f}%")
print(f"  Median residual:     {result['median_residual']:.4f} dex")
print(f"  Wilcoxon statistic:  {result['wilcoxon_stat']:.1f}")
print(f"  Wilcoxon p-value:    {result['wilcoxon_pvalue']:.4f}")

## 5. g(Rt) vs M_bar with BTFR Trend

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

log_m = np.log10(mbar_array)
log_g = np.log10(g_array)

sc = ax.scatter(log_m, log_g, c=merged["Rt_over_Rmax"].values,
                cmap="viridis", edgecolors="k", linewidth=0.3, s=40, alpha=0.8)

# BTFR trend line
m_range = np.linspace(log_m.min() - 0.2, log_m.max() + 0.2, 100)
g_trend = BTFR_ALPHA * m_range + BTFR_BETA
ax.plot(m_range, g_trend, "r-", linewidth=2,
        label=rf"BTFR: $\alpha$={BTFR_ALPHA}, $\beta$={BTFR_BETA}")

# a0/2 reference
ax.axhline(np.log10(a0_half_mks), color="gray", linestyle=":", linewidth=1.5,
           label=f"$a_0/2$ = {a0_half_mks:.2e} m/s$^2$")

ax.set_xlabel(r"$\log_{10}(M_{\rm bar}$ / M$_\odot$)", fontsize=12)
ax.set_ylabel(r"$\log_{10}(g(R_t)$ / m s$^{-2}$)", fontsize=12)
ax.set_title("g(Rt) vs Baryonic Mass with Fixed BTFR Trend", fontsize=13)
ax.legend(fontsize=10, loc="upper left")
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label("$R_t / R_{max}$", fontsize=11)
plt.tight_layout()
plt.savefig(results_dir / "NB02_g_Rt_vs_Mbar.png", dpi=150)
plt.show()

## 6. Residual Distribution

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

residuals = result["residuals"]
ax.hist(residuals, bins=25, edgecolor="black", alpha=0.7, color="coral")
ax.axvline(0, color="black", linestyle="-", linewidth=1.5)
ax.axvline(result["median_residual"], color="orange", linestyle="--", linewidth=2,
           label=f"Median = {result['median_residual']:.3f} dex")

ax.set_xlabel(r"$\Delta \log_{10} g$ (residual from BTFR trend)", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(
    f"BTFR Residuals -- Wilcoxon p = {result['wilcoxon_pvalue']:.3f}, "
    f"scatter = {result['scatter_residual']:.3f} dex",
    fontsize=12,
)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(results_dir / "NB02_btfr_residuals.png", dpi=150)
plt.show()

## 7. Interpretation

The SPARC BTFR residuals are centered near zero **by construction** -- the intercept
beta was anchored to place the Paper 2 median at a0/2. This is expected and not
a confirmatory result.

What this notebook validates:
1. The BTFR residual pipeline (`src/btfr.py`) produces correct results on known data
2. The scatter reduction (raw -> residual) quantifies how much of the g(Rt) spread
   is explained by the mass-dependent BTFR trend
3. The pipeline is ready for **out-of-sample** application to THINGS (NB09) and
   LITTLE THINGS (NB13), where the Wilcoxon test becomes genuinely confirmatory